# Specs: frozen, versioned, hashed

Every configuration object in axiom inherits `Spec`. A `Spec` is immutable, rejects
unknown fields, serializes through one JSON envelope, hashes stably across processes, and
can diff itself against another of its type. This is what the provenance record in `io`
is built on: an analysis is the set of content hashes it touched.

In [ ]:
from axiom.core import (
    D,
    SchemaVersionError,
    Spec,
    SpecDiff,
    SpecError,
    TimeWindow,
    Treatment,
    UnknownSpecError,
    load_spec,
    spec_type_name,
)

## Defining one

Subclass `Spec` with pydantic fields. Validation runs once, at construction.

In [ ]:
class ProbeDesign(Spec):
    """A toy spec: which treatment to probe, at which dose levels, over which window."""

    treatment: Treatment
    levels: tuple[float, ...]
    window: TimeWindow
    replicate: int = 1


design = ProbeDesign(
    treatment=Treatment(name="fertilizer", dimension=D.currency, unit="USD"),
    levels=(0.0, 50.0, 100.0),
    window=TimeWindow(start=0, stop=8),
)
design

In [ ]:
try:
    design.replicate = 2
except Exception as e:  # pydantic raises ValidationError on a frozen model
    print(type(e).__name__, "- specs are immutable; use model_copy(update=...)")

design_2 = design.model_copy(update={"replicate": 2})
print(design_2.replicate)

## The envelope

`to_json()` writes `{"spec": "<module>:<Class>", "schema_version": ..., "data": {...}}`.
`from_json` resolves the class by *name*, never by executing stored code, and refuses to
load a different class than the one you asked for.

In [ ]:
print(spec_type_name(ProbeDesign))
print(design.to_json(indent=2)[:400], "...")

In [ ]:
back = ProbeDesign.from_json(design.to_json())
print("round-trips equal:", back == design)

generic = load_spec(design.to_json())   # when you do not know the class in advance
print("load_spec gives the concrete type:", type(generic).__name__)

try:
    TimeWindow.from_json(design.to_json())
except UnknownSpecError as e:
    print("wrong class refused:", e)

try:
    Spec.from_json('{"not": "an envelope"}')
except SpecError as e:
    print("bad envelope refused:", e)

## Content hashes

`content_hash()` is blake2b-256 over the canonical envelope (sorted keys, no whitespace,
floats via `repr`, `Fraction` as `"p/q"`). It does not depend on field order, dict
insertion order, `PYTHONHASHSEED`, or the process. Two specs are the same spec iff their
hashes match.

In [ ]:
h = design.content_hash()
print(h)
print("stable across copies:", design.model_copy().content_hash() == h)
print("changes with content:", design_2.content_hash() == h)
print("hash() works too, so specs can key dicts and sets:", hash(design) == hash(back))

## Diffs

`diff` returns a `SpecDiff` with dotted paths, which is what the assumption ledger uses to
say *which facet* of an estimand differed.

In [ ]:
d: SpecDiff = design.diff(
    design.model_copy(update={"levels": (0.0, 50.0, 150.0), "window": TimeWindow(start=0, stop=12)})
)
print(d.is_empty)
for path, (left, right) in d.changed.items():
    print(f"  {path}: {left!r} -> {right!r}")

## Schema versions and migrations

Each class carries a `SCHEMA_VERSION`. Loading a payload written at an older version either
applies a registered migration or raises `SchemaVersionError` naming both versions. It never
silently coerces — that is how saved analyses rot at the first field rename.

In [ ]:
class ProbeDesignV2(Spec):
    SCHEMA_VERSION = "2"
    treatment: Treatment
    levels: tuple[float, ...]
    window: TimeWindow
    replicates: int = 1          # renamed from `replicate`


# a payload written by the v1 class
v1_payload = {
    "spec": spec_type_name(ProbeDesignV2),
    "schema_version": "1",
    "data": {**design.to_dict()},
}
import json

try:
    ProbeDesignV2.from_json(json.dumps(v1_payload))
except SchemaVersionError as e:
    print("no migration:", e)

ProbeDesignV2.register_migration("1", "2", lambda d: {**{k: v for k, v in d.items() if k != "replicate"}, "replicates": d["replicate"]})
migrated = ProbeDesignV2.from_json(json.dumps(v1_payload))
print("migrated:", migrated.replicates, type(migrated).__name__)

## Discovery

`Spec.subclasses()` lists every spec class the process knows about. The round-trip gate
(`tests/contracts/test_spec_roundtrip.py`) walks this list and fails if any class lacks an
example in its factory table.

In [ ]:
[c.__name__ for c in Spec.subclasses() if c.__module__.startswith("axiom.core")]